In [1]:
import pandas as pd
import time
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

from src.clients.lastfm_client import LastFMClient
from src.config import LASTFM_API_KEY

In [7]:
DATA_PATH = Path("../data/processed/lastfm_scrobbles_clean.parquet")

In [8]:
# LOAD DATA
df = pd.read_parquet(DATA_PATH)

unique_artists = df["artist"].dropna().unique()

In [9]:
client = LastFMClient(api_key=LASTFM_API_KEY)

In [10]:
# FETCH TAGS FROM LASTFM
artist_tags_cache = {}

for i, artist in enumerate(unique_artists):
    
    if artist in artist_tags_cache:
        continue
    
    tags = client.get_artist_tags(artist)
    artist_tags_cache[artist] = tags
    
    if i % 50 == 0:
        print(i)
    
    time.sleep(0.3)

0
50
100
150
200
250
300
350
400
450
500
550
600
650
700
750
800
850
900
950
1000
1050
1100
1150
1200
1250
1300
1350
1400
1450
1500
1550
1600
1650
1700
1750
1800
1850
1900
1950
2000
2050
2100
2150
2200
2250
2300
2350
2400
2450
2500
2550
2600
2650
2700
2750
2800
2850
2900
2950
3000
3050
3100
3150
3200
3250
3300
3350
3400
3450
3500
3550
3600
3650
3700
3750
3800
3850
3900
3950
4000
4050
4100
4150
4200
4250
4300
4350
4400
4450
4500
4550
4600
4650
4700
4750
4800
4850
4900
4950
5000
5050
5100
5150
5200
5250
5300
5350
5400
5450
5500
5550
5600
5650
5700
5750
5800
5850
5900
5950
6000
6050
6100
6150
6200
6250
6300
6350
6400
6450
6500
6550
6600
6650
6700
6750
6800
6850
6900
6950
7000
7050
7100
7150
7200
7250
7300
7350
7400
7450
7500
7550
7600
7650
7700
7750
7800
7850
7900
7950
8000
8050
8100
8150
8200
8250
8300
8350
8400
8450
8500
8550
8600
8650
8700
8750
8800
8850
8900
8950
9000
9050
9100
9150
9200
9250
9300
9350
9400
9450
9500
9550
9600
9650
9700
9750
9800
9850
9900
9950
10000
10050
10100
10150

In [11]:
df["tags"] = df["artist"].map(artist_tags_cache)

In [15]:
# DROP DUPLICATES FOR MODELING
model_df = df.drop_duplicates(subset=["artist_clean", "track_clean"])

In [17]:
# CLEAN LASTFM TAGS

def clean_tags(tags):
    if not isinstance(tags, list):
        return []
    
    return list(set([t.lower() for t in tags if len(t) > 2 and "seen live" not in t]))

In [18]:
model_df["tags_clean"] = model_df["tags"].apply(clean_tags)

In [19]:
# SANITY CHECK

model_df["tags_clean"].explode().value_counts().head(20)

tags_clean
indie                33928
alternative          30110
rock                 29715
indie rock           21489
american             20594
electronic           19089
british              17961
alternative rock     17733
female vocalists     15384
pop                  15281
singer-songwriter    14370
indie pop            13707
experimental         13061
folk                 12914
ambient               9639
dream pop             7797
post-punk             7291
polish                5956
electronica           5729
psychedelic           5477
Name: count, dtype: int64

In [ ]:
model_df.to_parquet("../data/processed/lastfm_scrobbles_with_tags.parquet", index=False)